# Slip–coupling sampling: 2007 $M_w$ 7.7 Tocopilla, Chile

This notebook shows how to sample a finite-fault slip model and compute the slip–coupling correlation, following the workflow of Oryan & Gabriel, *Do Coupled Megathrusts Rupture?* (preprint, [doi:10.31223/X5HB3N](https://doi.org/10.31223/X5HB3N)). It reproduces the Tocopilla example in Fig. 2 and walks through the main steps:

1. load a kinematic coupling model and a finite-fault slip model,
2. represent the slip distribution as a slip-weighted point cloud (one sampling density κ),
3. assign coupling to every sample and compute the slip-weighted coupling CDF,
4. choose a sampling density κ for which the CDF has converged,
5. build a null model by randomly relocating and rotating the rupture within the margin,
6. compute CDFs for many realizations efficiently with `cdf_pdf`.

You are welcome to apply this code to your own earthquakes and coupling models. All coupling models are available for download from the **Coupling Cloud** ([couplingcloud.ucsd.edu](https://couplingcloud.ucsd.edu); Oryan et al., 2026, *Seismica*, [doi:10.26443/seismica.v5i1.2314](https://doi.org/10.26443/seismica.v5i1.2314)) and finite-fault models from **SRCMOD** (Mai & Thingbaijam, 2014, [equake-rc.info/srcmod](http://equake-rc.info/srcmod/)). Any Coupling Cloud `.nc` file and SRCMOD `.fsp` file can replace the two used here.

**Data in this folder**
- `Metois.nc`: coupling model of Métois et al. (2016), from the Coupling Cloud.
- `s2007TOCOPI01HAYE.fsp`: Tocopilla finite-fault model (Hayes, 2017), from SRCMOD.

Run this notebook from `examples/`.

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath(".."))          # repo root

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import cartopy.crs as ccrs
from matplotlib.ticker import PercentFormatter

import couplingPlotter as cp
import eventLoader
import finiteFaultSampler as ffs
import cdf_pdf

## 1. Load coupling and slip

In [ ]:
KAPPA = 3500                         # sampling density κ [km⁻³] (paper value)
ffs.SLIP_PER_AREA = KAPPA / 1e3      # .fsp slip is in m, area in km² → κ/1000

eq = eventLoader.ReadFSPFile("s2007TOCOPI01HAYE.fsp", extractWebpage=False)

ds = xr.open_dataset("Metois.nc")                  # Métois et al. (2016)
coupling = cp.CouplingObjectGrid(lat=ds.lat.values, lon=ds.lon.values,
                                 coupling=ds.coupling.transpose("lat", "lon").values)

# map plotters: whole margin, and zoom on the rupture
plotter = cp.CouplingMapPlotter([coupling], load_hillshade=False)
x0, x1, y0, y1 = eq.segments[0].extent
plotter_zoom = cp.CouplingMapPlotter([coupling], lon_min=x0-0.5, lon_max=x1+0.5, lat_min=y0-0.5, lat_max=y1+0.5,
                                     style_params={"xtick_major": 1, "ytick_major": 1, "xtick_minor": 0.2, "ytick_minor": 0.2},
                                     load_hillshade=False)

print(eq.metadata["Event"], eq.metadata["Date"], f"Mw {eq.metadata['Mw']}")

In [ ]:
pc = ccrs.PlateCarree()
fig, ax = plt.subplots(1, 2, figsize=(11, 7), subplot_kw={"projection": pc})

# A – coupling along the margin
plotter.create_map(ax=ax[0], hillshade=False)
ax[0].scatter(eq.metadata["long"], eq.metadata["lat"], marker="*", s=150, c="cyan", ec="k", transform=pc, zorder=5)
fig.colorbar(plt.cm.ScalarMappable(cmap="hot_r", norm=plt.Normalize(0, 1)), ax=ax[0], shrink=0.6, label="Coupling")

# B – finite-fault slip with coupling contours
plotter_zoom.create_map(ax=ax[1], plotCoupling=False, hillshade=False)
slip = eq.segments[0].Subfaults_collection(cmap="gnuplot2_r", edgecolor="k", linewidth=0.2, transform=pc)
ax[1].add_collection(slip)
coupling.plot_contour(ax=ax[1], contour_params={"levels": [0.1, 0.3, 0.5, 0.7, 0.9], "colors": "k", "linewidths": 0.8})
fig.colorbar(slip, ax=ax[1], shrink=0.6, label="Slip (m)")
plt.show()

## 2. Sample the rupture (one κ)

Each subfault $i$ gets $N_i$ points drawn uniformly within it, proportional to its area $A_i$ and slip $S_i$, with sampling density κ:

$$N_i = \kappa \, A_i S_i \tag{1}$$

Each sample $j$ then takes the coupling at its surface position, $C_j = C(x_j, y_j)$.

In [ ]:
sampler = coupling.ReturnCouplingSampler()

def sample_coupling(ev):
    xy = ev.Sample()                                                  # Eq. 1: N_i = κ A_i S_i points per subfault → (N_f, 2) lon/lat
    c = sampler.SampleValue(xy, denormalize_points=False).numpy()     # C_j = C(x_j, y_j)
    return xy.numpy(), c

xy, C = sample_coupling(eq)
print(f"N_f = {len(C):,}  (κ = {KAPPA} km⁻³, ΣA·S = {eq.TotalSlipArea()/1e3:.1f} km³)")

In [ ]:
from matplotlib.lines import Line2D

SLIP_LEVELS = [1, 2, 3]                                       # slip contours [m]
fig, ax = plt.subplots(figsize=(6, 7), subplot_kw={"projection": pc})
plotter_zoom.create_map(ax=ax, hillshade=False)               # coupling
eq.PlotContours(n_levels=SLIP_LEVELS, ax=ax, contour_kwargs={"colors": "blue", "linewidths": 1.5, "transform": pc})
ax.scatter(xy[:, 0], xy[:, 1], s=0.1, c="cyan", alpha=0.6, transform=pc)

ax.legend(handles=[Line2D([], [], color="blue", lw=1.5, label=f"Slip contours {SLIP_LEVELS} m"),
                   Line2D([], [], color="cyan", marker="o", ls="", ms=4, label=f"Samples ($N_f$ = {len(C):,})")],
          loc="lower left", fontsize=9)
fig.colorbar(plt.cm.ScalarMappable(cmap="hot_r", norm=plt.Normalize(0, 1)), ax=ax, shrink=0.6, label="Coupling")
plt.show()

## 3. Slip-weighted coupling CDF

Because samples are slip-weighted, the empirical CDF of the $N_t$ sampled coupling values describes how slip is distributed across coupling (Eq. 2):

$$\hat{P}(C) = \frac{1}{N_t}\sum_{q=1}^{N_t} \mathbf{1}_{\{C_q \le C\}} \tag{2}$$

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.ecdfplot(y=C, ax=ax, color="k", lw=2)                  # Eq. 2: empirical CDF computed and plotted by seaborn
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Cumulative slip", ylabel="Coupling")
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.grid(ls=":", alpha=0.5)
plt.show()

## 4. Choosing κ

κ sets how many samples represent the rupture. If κ is too small, the point cloud is a coarse version of the slip distribution and the CDF is noisy; if it is too large, computations get expensive (especially for the null model). Below, κ = 5 km⁻³ gives only ~10 samples (because $N_i$ is rounded down, only the highest-slip subfaults get any), while κ = 1000 km⁻³ (~10,000 samples) resolves the slip distribution well.

In [ ]:
def sample_at(kappa):
    eq.ChangeSlipPerArea(kappa / 1e3)          # same unit conversion as ffs.SLIP_PER_AREA
    return sample_coupling(eq)

fig = plt.figure(figsize=(11, 6))
ax_map = fig.add_subplot(1, 2, 1, projection=pc)
ax_cdf = fig.add_subplot(1, 2, 2)
plotter_zoom.create_map(ax=ax_map, hillshade=False)

# dense first (cyan, small), sparse on top (blue, large)
for kappa, col, size in [(1000, "cyan", 0.3), (5, "blue", 25)]:
    xy_k, c_k = sample_at(kappa)
    ax_map.scatter(xy_k[:, 0], xy_k[:, 1], s=size, color=col, ec="w" if size > 1 else "none", lw=0.5,
                   transform=pc, label=f"κ = {kappa} km⁻³ ($N_f$ = {len(c_k):,})")
    sns.ecdfplot(y=c_k, ax=ax_cdf, color=col, lw=2, label=f"κ = {kappa} km⁻³")
sns.ecdfplot(y=C, ax=ax_cdf, color="k", lw=1, ls="--", label=f"κ = {KAPPA} km⁻³")
ax_map.legend(loc="lower left", fontsize=8, markerscale=3)

ax_cdf.set(xlim=(0, 1), ylim=(0, 1), xlabel="Cumulative slip", ylabel="Coupling")
ax_cdf.xaxis.set_major_formatter(PercentFormatter(1))
ax_cdf.legend(loc="upper left")
ax_cdf.grid(ls=":", alpha=0.5)
plt.show()

To pick κ (Supplementary Text S2), compute the CDF for a range of κ and compare each with a dense reference ($\kappa_{\mathrm{ref}} = 10^5$ km⁻³, ~1 M samples here):

$$\Delta_\kappa = \max_C \left| \hat{P}_\kappa(C) - \hat{P}_{\mathrm{ref}}(C) \right|$$

and take the smallest κ above which $\Delta_\kappa$ stays below a tolerance (1 %). In the paper, κ = 3500 km⁻³ keeps $\Delta_\kappa$ < 1 % for all 61 events (750 km⁻³, < 5 %, for the null model); small events need larger κ than Tocopilla.

In [ ]:
c_grid = np.linspace(0, 1, 201)
kappas = np.logspace(1, 4.5, 8)                           # 10 … ~3·10⁴ km⁻³
cdf_ref = cdf_pdf.ecdf_on_grid(sample_at(1e5)[1], c_grid, force_endpoints=False)

cdfs, n_f = [], []
for kappa in kappas:
    _, c_k = sample_at(kappa)
    cdfs.append(cdf_pdf.ecdf_on_grid(c_k, c_grid, force_endpoints=False))
    n_f.append(len(c_k))
eq.ChangeSlipPerArea(KAPPA / 1e3)                         # back to the working κ

delta = np.abs(np.array(cdfs) - cdf_ref)                  # (n_kappa, n_grid)
delta_max = delta.max(axis=1)
TOL = 0.01
ok = delta_max < TOL
kappa_min = kappas[next((i for i in range(len(kappas)) if ok[i:].all()), len(kappas) - 1)]  # Δ stays < TOL above this κ
print(f"smallest κ with Δ < {TOL:.0%}: {kappa_min:,.0f} km⁻³")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
cmap = plt.cm.viridis(np.linspace(0, 1, len(kappas)))
for d, kappa, col in zip(np.array(cdfs) - cdf_ref, kappas, cmap):
    ax.plot(d, c_grid, color=col, lw=1.2, label=f"{kappa:,.0f}")
ax.axvspan(-TOL, TOL, color="grey", alpha=0.2, label=f"±{TOL:.0%}")
ax.set(xlabel=r"$\hat{P}_\kappa - \hat{P}_{ref}$", ylabel="Coupling", ylim=(0, 1))
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.legend(title="κ (km⁻³)", fontsize=7, ncol=2)
ax.grid(ls=":", alpha=0.5)
plt.show()

## 5. Null model: randomly relocated ruptures

To test whether the observed slip–coupling correlation is meaningful, we compare it against a null model in which the spatial association between slip and coupling is random. Each null realization keeps the slip distribution of the real event but places it elsewhere on the margin:

- translate the rupture so its epicenter falls on a random point where coupling is defined,
- rotate it about the new epicenter by an angle drawn uniformly from 0–360°,
- keep the realization only if ≥ 95 % of its samples fall on defined coupling.

The null CDFs then reflect the margin's baseline coupling, i.e. the slip–coupling relation expected by chance. Here we draw `N_NULL` = 5 realizations; the paper uses 500 per event and compares observed and null CDFs pairwise.

In [ ]:
N_NULL = 5                                    # number of null realizations
MAX_TRIES = 500                               # stop after this many attempts
rng = np.random.default_rng(0)
lon2d, lat2d = coupling.GetLon2dLat2d()
defined = np.isfinite(coupling.data_array.values)
candidates = np.column_stack([lon2d[defined], lat2d[defined]])

moved = []                                    # (event, samples, coupling)
for tries in range(1, MAX_TRIES + 1):
    lon0, lat0 = candidates[rng.integers(len(candidates))]
    ev = eq.ShiftEQ(float(lon0), float(lat0)).RotateAroundPoint(rng.uniform(0, 360))
    xy_m, c_m = sample_coupling(ev)
    if np.isfinite(c_m).mean() >= 0.95:
        moved.append((ev, xy_m, c_m))
    if len(moved) == N_NULL:
        break
print(f"accepted {len(moved)} realizations in {tries} tries")

In [ ]:
colors = sns.color_palette("tab10", len(moved))
fig = plt.figure(figsize=(11, 7))
ax_map = fig.add_subplot(1, 2, 1, projection=pc)
ax_cdf = fig.add_subplot(1, 2, 2)

plotter.create_map(ax=ax_map, hillshade=False)
ax_map.scatter(xy[::20, 0], xy[::20, 1], s=0.5, c="cyan", transform=pc)
for (ev, xy_m, c_m), col in zip(moved, colors):
    ax_map.scatter(xy_m[::20, 0], xy_m[::20, 1], s=0.5, color=col, transform=pc)

t0 = time.perf_counter()
for (ev, xy_m, c_m), col in zip(moved, colors):
    sns.ecdfplot(y=c_m, ax=ax_cdf, color=col, lw=1)
T_ECDF = time.perf_counter() - t0                 # seaborn ECDF time for the null realizations
sns.ecdfplot(y=C, ax=ax_cdf, color="k", lw=2.5, label="observed")

ax_cdf.set(xlim=(0, 1), ylim=(0, 1), xlabel="Cumulative slip", ylabel="Coupling")
ax_cdf.xaxis.set_major_formatter(PercentFormatter(1))
ax_cdf.legend(loc="upper left")
ax_cdf.grid(ls=":", alpha=0.5)
plt.show()

## 6. Fast CDFs with `cdf_pdf`

`CDF_per_columns` evaluates Eq. 2 on a fixed coupling grid for every column of an $(N_t \times k)$ array in chunks (dask), which is how the 500 realizations are handled in the paper. With one κ all realizations have the same $N_t$, so they stack into columns. The timing below compares it with the seaborn ECDFs of the previous block; the speed-up grows with the number of realizations, and with `N_NULL` = 500 (as in the paper) `cdf_pdf` is an order of magnitude faster, if not more.

In [ ]:
C_null = np.column_stack([c_m for _, _, c_m in moved])                 # (N_t, N_NULL)

t0 = time.perf_counter()
cdf_null = cdf_pdf.CDF_per_columns(C_null, c_grid).computeCDFsForSubset(returnXarray=True, progress="none")  # dims (x, k)
T_CDF = time.perf_counter() - t0
print(f"cdf_pdf: {T_CDF:.3f} s   seaborn ecdf: {T_ECDF:.3f} s   →  {T_ECDF / T_CDF:.1f}× faster")
cdf_obs = cdf_pdf.ecdf_on_grid(C, c_grid, force_endpoints=False)

fig, ax = plt.subplots(figsize=(5, 4))
for k, col in zip(cdf_null.k.values, colors):
    ax.plot(cdf_null.sel(k=k).values, cdf_null.x.values, color=col, lw=1)
ax.plot(cdf_obs, c_grid, color="k", lw=2.5, label="observed")
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Cumulative slip", ylabel="Coupling")
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.legend(loc="upper left")
ax.grid(ls=":", alpha=0.5)
plt.show()

---
All data for the paper are available in *Supplements for Do Coupled Megathrusts Rupture?*, [doi:10.5281/zenodo.22836382](https://doi.org/10.5281/zenodo.22836382).